# 01 — Data Collection

This notebook drives the **raw-data acquisition** stage of the project:

1. Download IMDB official datasets (`.tsv.gz`).
2. Download & extract MovieLens 25M.
3. Inspect schemas, row counts, and file sizes.
4. Filter `title.basics` to feature films only.
5. Run a small TMDB enrichment sample.
6. Produce a **data-quality report** (nulls, duplicates, anomalies).

> All downloads are idempotent — re-running the notebook does not re-fetch
> files that already exist on disk.

## 0. Setup

In [ ]:
# Make project root importable from a notebook in ./notebooks/
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from config import settings, setup_logging
setup_logging()

import logging
logger = logging.getLogger("01_data_collection")

logger.info("Project root: %s", settings.PATHS.ROOT)
logger.info("Raw-data dir: %s", settings.PATHS.RAW_DATA)
logger.info("TMDB configured: %s", settings.TMDB.is_configured())

## 1. Download IMDB datasets

We pull all five official IMDB TSVs. They stay gzipped on disk — pandas reads
`.tsv.gz` directly.

In [ ]:
from src.data.downloader import IMDBDownloader

imdb_files = IMDBDownloader().download_all()
for name, path in imdb_files.items():
    size_mb = path.stat().st_size / 1e6
    logger.info("%-30s %7.1f MB  ->  %s", name, size_mb, path)

## 2. Download MovieLens 25M

This is a ~250 MB zip; first run takes a few minutes. Subsequent runs are
free (skip-if-extracted).

In [ ]:
from src.data.downloader import MovieLensDownloader

ml_dir = MovieLensDownloader().download_and_extract()
logger.info("MovieLens extracted at: %s", ml_dir)
for csv in sorted(ml_dir.glob("*.csv")):
    logger.info("  %-15s %7.1f MB", csv.name, csv.stat().st_size / 1e6)

## 3. Schema inspection — IMDB

For each TSV we read just the **first 1000 rows** to discover the schema
cheaply. We also report the *full* row count using a streamed line count.

In [ ]:
import pandas as pd

def quick_schema(path, n=5):
    df = pd.read_csv(path, sep="\t", nrows=1000, na_values="\\N",
                     low_memory=False)
    return {
        "columns": list(df.columns),
        "dtypes": {c: str(t) for c, t in df.dtypes.items()},
        "sample": df.head(n).to_dict(orient="records"),
    }

schemas = {name: quick_schema(path) for name, path in imdb_files.items()}
for name, info in schemas.items():
    print(f"\n== {name} ==")
    print(" columns:", info["columns"])

In [ ]:
# Row counts (streamed — no full read into memory)
import gzip

def gz_line_count(path):
    n = 0
    with gzip.open(path, "rb") as fh:
        for _ in fh:
            n += 1
    return n - 1   # minus header

row_counts = {name: gz_line_count(path) for name, path in imdb_files.items()}
pd.Series(row_counts, name="rows").sort_values(ascending=False).to_frame()

## 4. Filter to feature films

`title.basics` contains every kind of title (TV, episodes, shorts…).
For our recommender we only want **movies** that survive our quality filters
(see `config.IMDBConfig`).

We use `usecols` + a `dtype` map so the full file fits in ~250 MB of RAM
rather than 1.5 GB.

In [ ]:
basics_path = imdb_files["title.basics.tsv.gz"]

USECOLS = ["tconst", "titleType", "primaryTitle", "originalTitle",
           "isAdult", "startYear", "runtimeMinutes", "genres"]
DTYPES = {
    "tconst": "string",
    "titleType": "category",
    "primaryTitle": "string",
    "originalTitle": "string",
    "isAdult": "string",          # mixed values — coerce after
    "startYear": "string",
    "runtimeMinutes": "string",
    "genres": "string",
}

basics = pd.read_csv(
    basics_path, sep="\t", na_values="\\N",
    usecols=USECOLS, dtype=DTYPES, low_memory=False,
)
logger.info("title.basics raw: %d rows, %.1f MB",
            len(basics), basics.memory_usage(deep=True).sum() / 1e6)

# numeric coercion (vectorized)
basics["startYear"] = pd.to_numeric(basics["startYear"], errors="coerce")
basics["runtimeMinutes"] = pd.to_numeric(basics["runtimeMinutes"], errors="coerce")
basics["isAdult"] = pd.to_numeric(basics["isAdult"], errors="coerce").fillna(0).astype("int8")

# filter to feature films within sensible bounds
mask = (
    basics["titleType"].isin(settings.IMDB.TITLE_TYPE_FILTER)
    & basics["startYear"].between(settings.IMDB.MIN_YEAR, settings.IMDB.MAX_YEAR)
    & basics["runtimeMinutes"].between(settings.IMDB.MIN_RUNTIME_MIN,
                                       settings.IMDB.MAX_RUNTIME_MIN)
    & (basics["isAdult"] == 0)
)
movies = basics.loc[mask].reset_index(drop=True)
logger.info("After filtering: %d movies (%.1f%% of raw)",
            len(movies), 100 * len(movies) / len(basics))
movies.head()

## 5. Join with ratings

`title.ratings` gives us `averageRating` and `numVotes`. We **inner-join** on
`tconst` and additionally require `numVotes >= IMDBConfig.MIN_VOTES` to drop
obscure titles whose ratings are statistically unreliable.

In [ ]:
ratings_path = imdb_files["title.ratings.tsv.gz"]

ratings = pd.read_csv(
    ratings_path, sep="\t", na_values="\\N",
    dtype={"tconst": "string", "averageRating": "float32", "numVotes": "int32"},
)
logger.info("title.ratings: %d rows", len(ratings))

movies_rated = movies.merge(ratings, on="tconst", how="inner")
logger.info("After ratings inner-join: %d rows", len(movies_rated))

movies_rated = movies_rated.loc[
    movies_rated["numVotes"] >= settings.IMDB.MIN_VOTES
].reset_index(drop=True)
logger.info("After min-votes filter (>= %d): %d rows",
            settings.IMDB.MIN_VOTES, len(movies_rated))
movies_rated.describe(include="all").T.head(10)

## 6. TMDB enrichment — sample

We don't enrich every movie here (that's a long job — done in step 3 of the
project). Instead we **fetch ~20 random samples** to confirm the pipeline
works end-to-end and to give us a feel for what the joined data looks like.

In [ ]:
if not settings.TMDB.is_configured():
    print("Skipping TMDB sample — set TMDB_API_KEY in .env to enable.")
else:
    from src.data.tmdb_fetcher import TMDBFetcher

    sample_ids = (
        movies_rated.sort_values("numVotes", ascending=False)
                    .head(20)["tconst"].tolist()
    )
    fetcher = TMDBFetcher()
    sample_df = fetcher.fetch_many(sample_ids)
    sample_df[["imdb_id", "title", "production_countries",
               "budget", "revenue", "popularity",
               "vote_average", "vote_count"]]

## 7. Data-quality report

In [ ]:
def quality_report(df: pd.DataFrame, name: str) -> pd.DataFrame:
    null_pct = (df.isna().mean() * 100).round(2)
    duplicates = int(df.duplicated().sum())
    report = pd.DataFrame({
        "dtype":       df.dtypes.astype(str),
        "null_count":  df.isna().sum(),
        "null_pct":    null_pct,
        "n_unique":    df.nunique(dropna=True),
    })
    print(f"=== {name} ===  rows={len(df):,}  duplicates={duplicates}")
    return report

quality_report(movies_rated, "movies (filtered + rated)")

In [ ]:
# Sanity ranges
sanity = pd.DataFrame({
    "startYear":      [movies_rated["startYear"].min(),      movies_rated["startYear"].max()],
    "runtimeMinutes": [movies_rated["runtimeMinutes"].min(), movies_rated["runtimeMinutes"].max()],
    "averageRating":  [movies_rated["averageRating"].min(),  movies_rated["averageRating"].max()],
    "numVotes":       [movies_rated["numVotes"].min(),       movies_rated["numVotes"].max()],
}, index=["min", "max"]).T
sanity

In [ ]:
# Persist a small "collection summary" alongside the raw files so the
# downstream notebook can verify reproducibility.
import json
summary = {
    "imdb_files":      {k: str(v) for k, v in imdb_files.items()},
    "imdb_row_counts": row_counts,
    "movies_after_filter": int(len(movies)),
    "movies_after_rating_join_and_min_votes": int(len(movies_rated)),
}
out = settings.PATHS.PROCESSED_DATA / "01_collection_summary.json"
out.write_text(json.dumps(summary, indent=2))
logger.info("Wrote %s", out)
summary

## ✅ Step 2 outcomes

* IMDB & MovieLens raw files present in `data/raw/`.
* `title.basics` filtered to **feature films** with sensible runtime / year /
  votes guards.
* TMDB client smoke-tested on a small sample.
* A reproducible **collection summary** persisted to
  `data/processed/01_collection_summary.json`.

Next: **Step 3 — Data cleaning & feature engineering** (`02_data_cleaning.ipynb`).